# Business-birth prediction model: Métropole du Grand Paris

This notebook rebuilds the existing BPE + SIDE model for all municipalities in the Métropole du Grand Paris: Paris and departments 92, 93, and 94.

The 2025 target is used only for holdout validation. The base panel must contain all target municipalities and the historical death variables; a department-93-only panel is rejected by the coverage checks below.

In [1]:
# =============================================================================
# GREATER PARIS DATA DOWNLOAD / PREPARATION ONLY
# =============================================================================
# This stage loads the existing raw files, keeps only the Métropole du Grand
# Paris municipalities, and writes inspectable files before any ML is run.

import json
import os

import pandas as pd

BASE_ROOT = r"D:\interview - Cergy Paris\paris_urban_data\paris_urban_data"
BASE_RAW = os.path.join(BASE_ROOT, "data", "raw")
BPE_DIR = os.path.join(BASE_RAW, "bpe")
SIDE_DIR = os.path.join(BASE_RAW, "side")

OUTPUT_DIR = r"D:\interview - Cergy Paris\greater paris"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BPE_EVOL_CSV = os.path.join(BPE_DIR, "ds_bpe_evolution_com_2015_2025_geo_2026.csv")
TYPEQU_CSV = os.path.join(BPE_DIR, "TYPEQU_2025.csv")
PASSAGE_CSV = os.path.join(BPE_DIR, "BPE25_table_passage.csv")
SIDE_ENT = os.path.join(SIDE_DIR, "TAB_SIDE_CREA_ENT_COM_HISTO_2025.xlsx")
SIDE_ETAB = os.path.join(SIDE_DIR, "TAB_SIDE_CREA_ETAB_COM_HISTO_2025.xlsx")

# Métropole du Grand Paris: Paris, Hauts-de-Seine, Seine-Saint-Denis,
# and Val-de-Marne.
DEPT_PREFIXES = ("75", "92", "93", "94")
YEARS_TO_KEEP = (2013, 2015, 2016, 2019, 2020, 2021, 2024, 2025)


def clean_code(series):
    values = series.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    return values.apply(
        lambda value: value.zfill(5) if value.isdigit() and len(value) < 5 else value
    )


def keep_greater_paris(df, code_column):
    df = df.copy()
    df[code_column] = clean_code(df[code_column])
    return df[df[code_column].str[:2].isin(DEPT_PREFIXES)].copy()


def find_code_column(df):
    normalized = {
        str(column).strip().lower().replace(" ", "").replace("_", "").replace("-", ""): column
        for column in df.columns
    }
    candidates = [
        "Code géographique",
        "CODEGEO",
        "CODGEO",
        "code_commune",
    ]
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
        key = candidate.lower().replace(" ", "").replace("_", "").replace("-", "")
        if key in normalized:
            return normalized[key]
    raise KeyError(f"Could not identify commune-code column: {list(df.columns)}")


def load_side_sheet(path, label):
    workbook = pd.ExcelFile(path)
    for sheet in workbook.sheet_names:
        preview = pd.read_excel(path, sheet_name=sheet, header=None, dtype=str)
        for header_row in range(min(20, len(preview))):
            row = preview.iloc[header_row].astype(str)
            if row.str.contains("Code", case=False, na=False).any():
                data = pd.read_excel(
                    path,
                    sheet_name=sheet,
                    header=header_row,
                    dtype=str,
                )
                data.columns = [str(column).strip() for column in data.columns]
                print(f"{label}: sheet={sheet!r}, header row={header_row}")
                return data
    raise RuntimeError(f"Could not find a SIDE data header in {path}")


def save_coverage(df, code_column, output_name):
    coverage = (
        df[code_column]
        .astype(str)
        .str[:2]
        .value_counts()
        .sort_index()
        .rename_axis("department")
        .reset_index(name="rows")
    )
    coverage.to_csv(os.path.join(OUTPUT_DIR, output_name), index=False)
    print(f"\n{output_name}")
    print(coverage.to_string(index=False))


# -----------------------------------------------------------------------------
# 1. Load and save BPE data for Greater Paris
# -----------------------------------------------------------------------------

bpe = pd.read_csv(
    BPE_EVOL_CSV,
    sep=None,
    engine="python",
    dtype={"GEO": str, "FACILITY_TYPE": str},
)
bpe.columns = bpe.columns.str.strip()
required_bpe = {"TIME_PERIOD", "GEO_OBJECT", "GEO", "FACILITY_TYPE", "OBS_VALUE"}
missing_bpe = required_bpe - set(bpe.columns)
if missing_bpe:
    raise KeyError(f"Missing BPE columns: {sorted(missing_bpe)}")

bpe["GEO"] = clean_code(bpe["GEO"])
bpe["TIME_PERIOD"] = pd.to_numeric(bpe["TIME_PERIOD"], errors="coerce")
bpe = bpe[bpe["GEO_OBJECT"].astype(str).str.strip() == "COM"]
bpe = bpe[bpe["GEO"].str[:2].isin(DEPT_PREFIXES)]
bpe = bpe[bpe["TIME_PERIOD"].isin(YEARS_TO_KEEP)].copy()

bpe_output = os.path.join(OUTPUT_DIR, "bpe_evolution_greater_paris.csv")
bpe.to_csv(bpe_output, index=False)
save_coverage(bpe, "GEO", "bpe_coverage_by_department.csv")
print(f"Saved BPE: {bpe_output}")
print(f"BPE rows: {len(bpe):,}")
print(f"BPE communes: {bpe['GEO'].nunique():,}")
print(f"BPE years: {sorted(bpe['TIME_PERIOD'].dropna().unique().tolist())}")


# -----------------------------------------------------------------------------
# 2. Copy domain mapping files into the Greater Paris data folder
# -----------------------------------------------------------------------------

typequ = pd.read_csv(TYPEQU_CSV, sep=None, engine="python", dtype=str)
passage = pd.read_csv(PASSAGE_CSV, sep=None, engine="python", dtype=str)
typequ.to_csv(os.path.join(OUTPUT_DIR, "TYPEQU_2025.csv"), index=False)
passage.to_csv(os.path.join(OUTPUT_DIR, "BPE25_table_passage.csv"), index=False)
print("Saved BPE domain mapping files.")


# -----------------------------------------------------------------------------
# 3. Load and save SIDE ENT and SIDE ETAB for Greater Paris
# -----------------------------------------------------------------------------

def prepare_side(path, label, output_name):
    data = load_side_sheet(path, label)
    code_column = find_code_column(data)
    data = keep_greater_paris(data, code_column)

    year_columns = [
        column
        for column in data.columns
        if str(column).strip().isdigit()
        and 2012 <= int(str(column).strip()) <= 2025
    ]
    keep_columns = [code_column] + year_columns
    data = data[keep_columns].copy()
    data = data.rename(columns={code_column: "code_commune"})
    data["code_commune"] = clean_code(data["code_commune"])
    data = data.rename(columns={column: int(str(column).strip()) for column in year_columns})

    for year in year_columns:
        data[int(str(year).strip())] = pd.to_numeric(
            data[int(str(year).strip())],
            errors="coerce",
        )

    output_path = os.path.join(OUTPUT_DIR, output_name)
    data.to_csv(output_path, index=False)
    save_coverage(data, "code_commune", f"{label.lower()}_coverage_by_department.csv")
    print(f"Saved SIDE {label}: {output_path}")
    print(f"SIDE {label} rows: {len(data):,}")
    print(f"SIDE {label} communes: {data['code_commune'].nunique():,}")
    return data


side_ent = prepare_side(
    SIDE_ENT,
    "ENT",
    "side_ent_greater_paris.csv",
)
side_etab = prepare_side(
    SIDE_ETAB,
    "ETAB",
    "side_etab_greater_paris.csv",
)


# -----------------------------------------------------------------------------
# 4. Save a small manifest so the data stage is easy to verify
# -----------------------------------------------------------------------------

manifest = {
    "geography": "Métropole du Grand Paris",
    "departments": list(DEPT_PREFIXES),
    "bpe_rows": int(len(bpe)),
    "bpe_communes": int(bpe["GEO"].nunique()),
    "side_ent_rows": int(len(side_ent)),
    "side_ent_communes": int(side_ent["code_commune"].nunique()),
    "side_etab_rows": int(len(side_etab)),
    "side_etab_communes": int(side_etab["code_commune"].nunique()),
    "output_directory": OUTPUT_DIR,
}
with open(os.path.join(OUTPUT_DIR, "greater_paris_data_manifest.json"), "w", encoding="utf-8") as manifest_file:
    json.dump(manifest, manifest_file, indent=2, ensure_ascii=False)

print("\nDATA STAGE COMPLETE")
print(json.dumps(manifest, indent=2, ensure_ascii=False))


bpe_coverage_by_department.csv
department  rows
        92 11376
        93 11412
        94 12882
Saved BPE: D:\interview - Cergy Paris\greater paris\bpe_evolution_greater_paris.csv
BPE rows: 35,670
BPE communes: 122
BPE years: [2015, 2020, 2025]
Saved BPE domain mapping files.
ENT: sheet='COM', header row=0

ent_coverage_by_department.csv
department  rows
        75     1
        92    36
        93    39
        94    47
Saved SIDE ENT: D:\interview - Cergy Paris\greater paris\side_ent_greater_paris.csv
SIDE ENT rows: 123
SIDE ENT communes: 123
ETAB: sheet='COM', header row=0

etab_coverage_by_department.csv
department  rows
        75     1
        92    36
        93    39
        94    47
Saved SIDE ETAB: D:\interview - Cergy Paris\greater paris\side_etab_greater_paris.csv
SIDE ETAB rows: 123
SIDE ETAB communes: 123

DATA STAGE COMPLETE
{
  "geography": "Métropole du Grand Paris",
  "departments": [
    "75",
    "92",
    "93",
    "94"
  ],
  "bpe_rows": 35670,
  "bpe_communes

In [2]:
# =============================================================================
# 5. EXPORT THE COMPLETE XGBOOST MODEL INPUT PACKAGE
# =============================================================================
# The model also uses historical births, establishments, and deaths from a
# processed panel. Export that panel separately; do not infer death values from
# BPE or SIDE creation files.

PANEL_CANDIDATES = [
    os.path.join(BASE_ROOT, "data", "processed", "merged_panel_grand_paris.parquet"),
    os.path.join(BASE_ROOT, "data", "processed", "merged_panel_idf.parquet"),
    os.path.join(BASE_ROOT, "data", "processed", "merged_panel_93.parquet"),
]

panel_source = next((path for path in PANEL_CANDIDATES if os.path.exists(path)), None)
if panel_source is None:
    print("No processed base panel was found.")
    print("The all-Greater-Paris panel must be created before ML can use death variables.")
    panel = None
else:
    print(f"Loading processed panel: {panel_source}")
    panel = pd.read_parquet(panel_source)
    if "code_commune" not in panel.columns:
        raise KeyError("Processed panel must contain code_commune.")

    panel["code_commune"] = clean_code(panel["code_commune"])
    panel = panel[panel["code_commune"].str[:2].isin(DEPT_PREFIXES)].copy()
    panel_output = os.path.join(OUTPUT_DIR, "base_panel_greater_paris.parquet")
    panel.to_parquet(panel_output, index=False)
    save_coverage(panel, "code_commune", "base_panel_coverage_by_department.csv")
    print(f"Saved base panel: {panel_output}")
    print(f"Panel rows: {len(panel):,}")
    print(f"Panel communes: {panel['code_commune'].nunique():,}")

    model_business_columns = [
        column
        for column in panel.columns
        if column.startswith(("n_births_", "n_establishments_", "n_deaths_"))
    ]
    if model_business_columns:
        panel["code_commune"].to_frame().join(
            panel[model_business_columns]
        ).to_csv(
            os.path.join(OUTPUT_DIR, "panel_business_variables_greater_paris.csv"),
            index=False,
        )
        print("Saved panel business and death variables.")

    panel_departments = set(panel["code_commune"].str[:2].unique())
    missing_departments = sorted(set(DEPT_PREFIXES) - panel_departments)
    if missing_departments:
        print(
            "WARNING: the available processed panel does not cover all Greater "
            f"Paris departments. Missing: {missing_departments}"
        )
        print(
            "BPE and SIDE are complete for the selected departments, but the "
            "XGBoost death features are not complete yet."
        )
    else:
        print("The processed panel covers all Greater Paris departments.")


# Create one inspectable business-demographics file from the two SIDE inputs.
# These are the exact creation-flow variables used by the model after renaming.
side_business = side_ent.merge(
    side_etab,
    on="code_commune",
    how="outer",
    suffixes=("_ent", "_etab"),
)
side_business.to_csv(
    os.path.join(OUTPUT_DIR, "business_demographics_greater_paris.csv"),
    index=False,
)
print("Saved combined SIDE business-demographics file.")


# Update the manifest with every data artifact required by the XGBoost notebook.
manifest.update(
    {
        "typequ_file": os.path.join(OUTPUT_DIR, "TYPEQU_2025.csv"),
        "passage_file": os.path.join(OUTPUT_DIR, "BPE25_table_passage.csv"),
        "bpe_file": bpe_output,
        "side_ent_file": os.path.join(OUTPUT_DIR, "side_ent_greater_paris.csv"),
        "side_etab_file": os.path.join(OUTPUT_DIR, "side_etab_greater_paris.csv"),
        "combined_business_file": os.path.join(
            OUTPUT_DIR, "business_demographics_greater_paris.csv"
        ),
        "base_panel_file": (
            os.path.join(OUTPUT_DIR, "base_panel_greater_paris.parquet")
            if panel is not None
            else None
        ),
        "death_features_available": bool(
            panel is not None
            and any(column.startswith("n_deaths_") for column in panel.columns)
        ),
        "ml_ready_for_all_departments": bool(
            panel is not None
            and set(panel["code_commune"].str[:2].unique()) >= set(DEPT_PREFIXES)
            and all(
                f"n_deaths_{year}" in panel.columns
                for year in (2013, 2016, 2019, 2021, 2024)
            )
        ),
    }
)
with open(
    os.path.join(OUTPUT_DIR, "greater_paris_data_manifest.json"),
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(manifest, manifest_file, indent=2, ensure_ascii=False)

print("\nCOMPLETE XGBOOST INPUT PACKAGE STATUS")
print(json.dumps(manifest, indent=2, ensure_ascii=False))

Loading processed panel: D:\interview - Cergy Paris\paris_urban_data\paris_urban_data\data\processed\merged_panel_93.parquet

base_panel_coverage_by_department.csv
department  rows
        93    40
Saved base panel: D:\interview - Cergy Paris\greater paris\base_panel_greater_paris.parquet
Panel rows: 40
Panel communes: 40
Saved panel business and death variables.
BPE and SIDE are complete for the selected departments, but the XGBoost death features are not complete yet.
Saved combined SIDE business-demographics file.

COMPLETE XGBOOST INPUT PACKAGE STATUS
{
  "geography": "Métropole du Grand Paris",
  "departments": [
    "75",
    "92",
    "93",
    "94"
  ],
  "bpe_rows": 35670,
  "bpe_communes": 122,
  "side_ent_rows": 123,
  "side_ent_communes": 123,
  "side_etab_rows": 123,
  "side_etab_communes": 123,
  "output_directory": "D:\\interview - Cergy Paris\\greater paris",
  "typequ_file": "D:\\interview - Cergy Paris\\greater paris\\TYPEQU_2025.csv",
  "passage_file": "D:\\interview